# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.

## Setup

In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [1]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

/workspaces/ai-agents-for-beginners/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Pattern 1: Clear Agent Instructions

The most impactful pattern is also the simplest: writing clear, detailed instructions for your agent.

Good instructions define:
- **Who** the agent is (persona and tone)
- **What** it should do (step-by-step responsibilities)
- **How** it should behave (constraints and style)

Below, we create a travel concierge agent with explicit instructions that shape every response it produces.

In [2]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

That sounds incredible! Exploring history while indulging in amazing cuisine is a fantastic way to immerse yourself in a destination. Based on your preferences and budget, I’ll work out some personalized options that balance culture, gastronomy, and a touch of luxury while staying within your $2,500 limit. Before diving into recommendations, let me ask a few questions to fine-tune your experience:

1. **Climate Preference:** Do you prefer warm, temperate, or cooler weather for this trip?
2. **Destination Preference:** Are you open to both domestic and international travel, or do you have a specific region in mind (Europe, Asia, South America, etc.)?
3. **Activity Level:** Do you like a mix of relaxed experiences and sightseeing, or are you open to walking tours and more dynamic activities?
4. **Travel Window:** When are you planning this trip? Knowing this will help narrow down the best destinations for your travel season.

Let me know, and I’ll craft the perfect historical and culinar

## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

We also introduce a tool that returns destination details so the agent grounds its recommendations in real data.

In [3]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

Here are three destinations ideal for a culture-loving traveler with a $2500 budget:

1. **Kyoto, Japan**
   - Known for its traditional wooden machiya houses, historic temples, and Zen gardens, Kyoto offers a glimpse into Japan's ancient culture. A visit during cherry blossom season or the fall foliage makes the city even more mesmerizing.

2. **Rome, Italy**
   - Experience a treasure trove of ancient history, art, and architecture in Rome. From the Colosseum and the Vatican City to its lively piazzas and authentic Italian cuisine, Rome is perfect for immersing yourself in European cultural heritage.

3. **Istanbul, Turkey**
   - This vibrant city bridges Asia and Europe, blending historic sites like the Hagia Sophia and Grand Bazaar with the flavors of Ottoman and Turkish cuisine. A cruise on the Bosphorus is a must to take in the city's architectural beauty.

These destinations can be explored within your budget, allowing for accommodations, sightseeing, and cultural experiences. L

## Pattern 3: Single Responsibility Agents

Complex tasks benefit from splitting work across multiple focused agents, each with a single responsibility:

- A **Destination Expert** that knows about places and availability
- A **Logistics Planner** that handles flights, hotels, and itineraries

This mirrors the software engineering principle of *separation of concerns* — each agent is easier to test, maintain, and improve independently.

In [4]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

=== Destination Expert ===
### Top 3 Recommendations for Culture and Food under $2500:

1. **Rome, Italy**
   - **Pros**: World-famous for art, architecture, and history along with authentic Italian cuisines like pasta and gelato. Rich in museums, historic landmarks such as the Colosseum, and unique cultural experiences.
   - **Cons**: Can feel tourist-heavy, and meal prices in popular areas may be higher.

2. **Istanbul, Turkey**
   - **Pros**: Amazing fusion of Eastern and Western cultures, with incredible street food, sprawling bazaars, historic mosques like the Hagia Sophia, and Turkish baths.
   - **Cons**: The city can get crowded, and some attractions may require long queues.

3. **Bangkok, Thailand**
   - **Pros**: Renowned for its vibrant street food scene, affordable fine dining, rich cultural temples, and bustling local markets.
   - **Cons**: Warm climate might not suit everyone, and navigating traffic can be challenging.

Let me know if you'd like to prioritize one!

=== L

## Summary

In this lesson we applied three agentic design patterns to a travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Define persona, responsibilities, and constraints up front | Consistent, on-brand agent behavior |
| **Structured Output** | Use Pydantic models as the response format | Validated, machine-readable results |
| **Single Responsibility** | Give each agent one focused job | Easier to test, maintain, and compose |

These patterns compose naturally — you can combine clear instructions with structured output inside a single-responsibility agent to build robust, production-ready systems.